# 01 · Baselines & teacher  (T4 · ~0.5 h)

1. **Baselines** — original notebook pipeline vs. fixed FaceNet vs. ArcFace iResNet-50 teacher, on the
   standard identity-disjoint verification benchmarks and on the LFW gallery (closed- and open-set).
2. **Teacher cache** — iResNet-50 embeddings for every CASIA image (distillation targets for notebook 02).

**Notebook settings:** Accelerator *GPU T4 ×1*, Internet *On*.
**Inputs:** the same datasets as notebook 00 **plus the output of notebook 00**.

In [ ]:
# Setup: clone the repo (code lives in src/fpgaface, notebooks are thin wrappers)
import os
REPO = "https://github.com/sidwastaken07/face-detection.git"
BRANCH = "claude/keen-meitner-ab0d8k"
CODE = "/tmp/face-detection"
if not os.path.exists(CODE):
    !git clone -q -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull -q
%cd {CODE}
!pip install -q -e . onnx onnx2torch
!pip install -q --no-deps facenet-pytorch

In [ ]:
import glob, os
DATA = os.path.dirname(glob.glob("/kaggle/input/**/data_manifest.json", recursive=True)[0])
print("prepared data:", DATA)

In [ ]:
!python scripts/baselines.py --data {DATA} --out /kaggle/working/results

In [ ]:
# Teacher embeddings for ~490k CASIA images (~15–20 min on a T4). Resumable if interrupted.
!python scripts/cache_teacher.py --data {DATA} --out /kaggle/working/teacher --teacher arcface_r50

In [ ]:
# Sanity check: cached teacher embeddings separate same vs different identities
import numpy as np, matplotlib.pyplot as plt
from fpgaface.utils import load_manifest
labels = np.load(load_manifest(DATA)["casia"]["labels"])
emb = np.load("/kaggle/working/teacher/teacher_arcface_r50.f16.npy").astype(np.float32)  # ~1 GB in RAM
rng = np.random.default_rng(0)
i, j = rng.integers(0, len(labels), (2, 200_000))            # random pairs: almost all different ids
k = rng.integers(0, len(labels) - 1, 100_000)
k = k[labels[k] == labels[k + 1]]                              # neighbours: images are grouped by identity
i, j = np.concatenate([i, k]), np.concatenate([j, k + 1])
same = labels[i] == labels[j]
s = np.einsum("nd,nd->n", emb[i], emb[j])
plt.figure(figsize=(6, 3))
plt.hist(s[same], 100, alpha=.6, density=True, label="same id")
plt.hist(s[~same], 100, alpha=.6, density=True, label="different id")
plt.legend(); plt.xlabel("cosine similarity"); plt.title("Teacher embeddings on CASIA"); plt.show()
